import libraries
prepare the data
split the data into:
    train
    validation
    test
add the model 
add randomized search parameters
save the best hyperparameters for the model
fit the tuned model to the training data
see the evaluation resutls on training dataset
check it on vlaidvalidation dataset 
- if perform well 
check it on test dataset
- if not: change the model hyperparameters and once satisfied 
check it on test dataset
plot the feature importance


In [18]:
# import necessary libraries
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

#preprocessing 
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
#modelselection
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV,train_test_split, KFold
#model 
from sklearn.ensemble import RandomForestClassifier
#evaluation metrics
from sklearn.metrics import accuracy_score, f1_score, precision_score, classification_report, confusion_matrix

In [5]:
df = pd.read_csv("../data/disease10K.csv")
df.head()


,age,bmi,blood_pressure,glucose_level,family_history,physical_activity_level,has_disease
0,24,21.5,108,90,0,low,0
1,70,24.5,124,105,0,moderate,0
2,62,29.0,118,124,0,high,0
3,47,24.0,140,129,0,low,0
4,47,26.0,140,114,1,moderate,0


SPLIT FIRST , THEN ENCODE
Split will be around train-validation-test, to do so we have to do 2 steps split first 70% training and 30% temporayr and then that 30 goes to validataion and test 

In [6]:

# features: columns except target
X = df.drop("has_disease",axis=1)
# target: column you want to classify
y = df["has_disease"]

In [ ]:
# FIRST CUT: Split out the Training set (80%)

X_train, X_temp, y_train,y_temp = train_test_split(
    X,y,test_size=0.20, stratify=y, random_state=42)
# SECOND CUT: Split the remaining 20% into Validation and Test (50/50)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

In [13]:
print("Train set ",X_train.shape)
print("Train set ",X_val.shape)
print("Train set ",X_test.shape)

Train set  (8000, 6)
Train set  (1000, 6)
Train set  (1000, 6)


In [10]:
# First split, then fit the encoder only on the training set.
# ⚠️ Remember, to avoid "cheating" (data leakage), you Fit on the Training set and only Transform the others.

encoder = OrdinalEncoder(categories=[["low", "moderate", "high"]])

X_train["physical_activity_level"] = encoder.fit_transform(
    X_train[["physical_activity_level"]]
)

X_val["physical_activity_level"] = encoder.transform(
    X_val[["physical_activity_level"]]
)

X_test["physical_activity_level"] = encoder.transform(
    X_test[["physical_activity_level"]]
)

In [15]:
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

In [20]:
# Parameter distributions for randomized search
param_dist = {
    "n_estimators": [50, 100, 150, 200, 300],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.3, 0.5],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [1, 2, 5, 10]
}



In [21]:
# randomized search
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring="accuracy",
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

In [22]:
# 7. Fit on training set only
random_search.fit(X_train,y_train)
# 8. Best model
best_model = random_search.best_estimator_

print("Best parameters:")
print(random_search.best_params_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters:
{'subsample': 1.0, 'reg_lambda': 2, 'reg_alpha': 0.01, 'n_estimators': 100, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.2, 'gamma': 0, 'colsample_bytree': 0.9}


In [24]:
y_train_pred = best_model.predict(X_train)


In [23]:
y_val_pred = best_model.predict(X_val)

In [25]:
print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))

print("\nValidation Classification Report:")
print(classification_report(y_val, y_val_pred))

Train Accuracy: 0.991625
Validation Accuracy: 0.988

Validation Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       986
           1       1.00      0.14      0.25        14

    accuracy                           0.99      1000
   macro avg       0.99      0.57      0.62      1000
weighted avg       0.99      0.99      0.98      1000



In [32]:
y_test_pred = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nTest Classification Report:")
print(classification_report(y_test, y_test_pred))

Test Accuracy: 0.987

Test Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       984
           1       0.71      0.31      0.43        16

    accuracy                           0.99      1000
   macro avg       0.85      0.66      0.71      1000
weighted avg       0.98      0.99      0.98      1000

